In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW=Path("../data/diabetic_data.csv")
OUT=Path("../data/diabetes_clean.csv")

In [2]:
df=pd.read_csv(RAW,na_values="?",keep_default_na=False)
print("Start:",df.shape)

C:\Users\hp\AppData\Local\Temp\ipykernel_19560\2285511271.py:1: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(RAW,na_values="?",keep_default_na=False)


Start: (101766, 50)


In [3]:
df[["max_glu_serum","A1Cresult"]].apply(lambda s:  s.value_counts(dropna=False))

,max_glu_serum,A1Cresult
>200,1485.0,NaN
>300,1264.0,NaN
>7,NaN,3812.0
>8,NaN,8216.0
None,96420.0,84748.0
Norm,2597.0,4990.0


In [4]:
not_eligible=[11,13,14,19,20,21]
before=len(df)
df=df[~df["discharge_disposition_id"].isin(not_eligible)]
print(f"Removed{before-len(df)}rows->{len(df)}")

Removed2423rows->99343


In [5]:
print(df["gender"].value_counts())
df=df[df["gender"]!="Unknown/Invalid"]

gender
Female             53454
Male               45886
Unknown/Invalid        3
Name: count, dtype: int64


In [6]:
before=len(df)
df=(df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr",keep="first"))

print(f"Removed {before-len(df)}repeat encounters->{len(df)}")
print("Unique patients== rows:",df["patient_nbr"].is_unique)

Removed 29353repeat encounters->69987
Unique patients== rows: True


In [7]:
drop_cols = ["encounter_id", "patient_nbr","weight","examide", "citoglipton"]      
df = df.drop(columns=drop_cols)
print(df.shape)

(69987, 45)


In [8]:
for col in ["race", "payer_code", "medical_specialty"]:
    df[col] = df[col].fillna("Missing")


for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].fillna("Missing")

print(df.isna().sum().sum(), "NaN values left")

0 NaN values left


In [9]:
id_cols = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
df[id_cols] = df[id_cols].astype(str)
df[id_cols].nunique()

admission_type_id            8
discharge_disposition_id    21
admission_source_id         17
dtype: int64

In [10]:
age_map = {f"[{i}-{i+10})": i + 5 for i in range(0, 100, 10)}
df["age_mid"] = df["age"].map(age_map)
print(df[["age", "age_mid"]].drop_duplicates().sort_values("age_mid"))

         age  age_mid
0     [0-10)        5
1    [10-20)       15
2    [20-30)       25
3    [30-40)       35
4    [40-50)       45
5    [50-60)       55
11   [60-70)       65
7    [70-80)       75
8    [80-90)       85
9   [90-100)       95


In [11]:
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
print(df["readmit_30"].value_counts(normalize=True).round(3))

readmit_30
0    0.91
1    0.09
Name: proportion, dtype: float64


In [12]:
assert df["readmit_30"].isin([0, 1]).all()
assert not df.duplicated().any()
assert df.isna().sum().sum() == 0

print(df.shape)
df.to_csv(OUT, index=False)
print("Saved to", OUT)

(69987, 47)
Saved to ..\data\diabetes_clean.csv


## Cleaning summary
- Loaded with `?` → NaN; kept "None" as a real category (test not performed).
- Removed died/hospice encounters (~2.4k rows) and 3 invalid-gender rows.
- Kept first encounter per patient (~70k rows) to prevent leakage.
- Dropped IDs, `weight`, and constant columns.
- Missing categoricals → "Missing".
- ID columns cast to categorical; `age_mid` added.
- Target `readmit_30` (1 = readmitted <30 days). Exclude `readmitted` from features.